In [ ]:
import pandas as pd
import  numpy as np

# read -> Ler
# dataframe -> Base de dados do Panda = df

df_vendas = pd.read_csv("vendas_tech.csv", low_memory=False) # encoding="latin1" --- sep=","
display(df_vendas)
df_gerente = pd.read_excel("gerentes_lojas.xlsx")
display(df_gerente)

In [35]:
# INSPEÇÃO
#display(df_vendas.head(10)) # head -> AS PRIMEIRAS LINHAS
#display(df_vendas.tail(10)) # tail -> AS ULTIMAS LINHAS
#display(df_vendas.sample(10)) # sample -> LINHAS ALEATORIAS

display(df_vendas.shape) # shape -> Para mostrar a quantidade de linhas e colunas

display(list(df_vendas.columns)) # Mostrar o nome das colunas

(100100, 8)

['ID_Pedido',
 'Data',
 'Loja',
 'Produto',
 'Preco_Unitario',
 'Qtd',
 'Cliente',
 'Data_Base']

In [ ]:
display(df_vendas.info())
display(df_vendas.describe())

In [ ]:
# TRATAMENRO DE DADOS
df_vendas["Loja"] # selecionando coluna

# colunas
df_analise = df_vendas.drop(columns=["Data_Base"]) # EXCLUIR LINHAS OU COLUNAS
#display(df_analise)

# nulos
df_analise = df_analise.dropna() # EXCLUI VALORES VAZIOS
#display(df_analise)

# tipos de dados
df_analise["Data"] = pd.to_datetime(df_analise["Data"], format="%Y-%m-%d")

# padronização
df_analise["Loja"] = df_analise["Loja"].str.strip() # Tirar os espaços
df_analise["Loja"] = df_analise["Loja"].str.title() # Padronizar os nomes

df_gerente["Loja"] = df_gerente["Loja"].str.strip() # Tirar os espaços
df_gerente["Loja"] = df_gerente["Loja"].str.title() # Padronizar os nomes

# duplicatas
df_analise = df_analise.drop_duplicates(subset=["ID_Pedido"])




#display(df_analise)
display(df_vendas.info())

In [ ]:
# CRIAR NOVAS COLUNAS

# FATURAMENTO
df_analise["Faturamento"] = df_analise["Qtd"] * df_analise["Preco_Unitario"]

# FORMA DE VENDA
df_analise["Forma_de_Venda"] = np.where(df_analise["Loja"]=="Online", "Online", "Presencial") # np = if -> Analisar uma coluna e tratalar

# REGIAO
#display(df_analise["Loja"].unique()) # Visualizar uma unica coluna

dic_regioes = {
    'São Paulo': "Sudeste",
    'Belo Horizonte': "Sudeste",
    'Online': "Online",
    'Rio De Janeiro': "Sudeste",
    'Salvador': "Nordeste",
    'Recife': "Nordeste",
    'Curitiba': "Sul",
    'Porto Alegre': "Sul"
    }

df_analise["Regiao"] = df_analise["Loja"].map(dic_regioes)



#display(df_analise)
display(df_analise.isna().sum())

In [32]:
# ANALISAR -> filtrar

df_analise = df_analise.sort_values(by=["Data", "Faturamento"], ascending=True) # Ordenando do mais antigo para o mais novo
df_analise = df_analise.reset_index(drop=True) # drop=True --> jogar o index antigo para fora

# .loc -> id do pedido --> por nome da coluna e por nome da linha
id_pedido = 4

loja = df_analise.loc[3, "Loja"]
produto = df_analise.loc[3, "Produto"]
cliente = df_analise.loc[3, "Cliente"]
print(loja, produto, cliente)

# comdicional
df_id_pedido_4 = df_analise[df_analise["ID_Pedido"]==4]

# exportar pedaços da base
df_vendas_sp = df_analise[df_analise["Loja"]=="São Paulo"]
df_vendas_sp.to_csv("Vendas_SP.csv", index=False) # TRANSFORMAR ESSA TABELA EM UM ARQUIVO CSV

# exportar as vendas de 2024
df_vendas_2024 = df_analise[df_analise["Data"].dt.year==2024]

# duplas condições
df_vendas_HDMI_SUL = df_analise[(df_analise["Produto"]=="Cabo HDMI") & (df_analise["Regiao"]=="Sul")]

#display(df_analise)
#display(df_id_pedido_4)
#display(df_vendas_sp)
#display(df_vendas_2024)
#display(df_vendas_HDMI_SUL)

Porto Alegre Cabo HDMI Cliente_22886


In [ ]:
# ANALISE POR AGRUPAMENTO
#display(df_analise)

# ranking de faturamento por loja
analise_lojas = df_analise[["Loja", "Faturamento"]].groupby("Loja").sum()
analise_lojas = analise_lojas.sort_values(by="Faturamento", ascending=True)
analise_lojas["Faturamento"] = analise_lojas["Faturamento"].map("R${:,.2f}".format) # CODIGO DE FORMATAÇÃO
display(analise_lojas)

# ranking de produtos que mais venderam no online
df_vendas_online = df_analise[df_analise["Loja"]=="Online"]

#analise de rankind por loja e por peoduto
# quais produtos venderam mais em cada loja
analise_produtos_em_lojas = df_analise[["Loja", "Produto", "Qtd"]].groupby(["Loja", "Produto"]).sum()

# quais lojas mais venderam os produtos
analise_lojas_em_produtos = df_analise[["Loja", "Produto", "Qtd"]].groupby(["Produto", "Loja"]).sum()

#display(analise_lojas_em_produtos)

In [ ]:
# quais gerentes bateram a meta em janeiro de 2023
df_vendas_gerentes_janeiro_23 = df_analise[(df_analise["Data"].dt.year==2023) & (df_analise["Data"].dt.month==1)]

df_vendas_gerentes_janeiro_23 = df_vendas_gerentes_janeiro_23[["Loja", "Faturamento"]].groupby("Loja", as_index=False).sum()

df_vendas_gerentes_janeiro_23 = df_vendas_gerentes_janeiro_23.merge(df_gerente, on="Loja", how="left") # juntar duas tabelas

df_vendas_gerentes_janeiro_23["Bateu a Meta"] = np.where(df_vendas_gerentes_janeiro_23["Faturamento"] >= df_vendas_gerentes_janeiro_23["Meta_Mensal"], "Sim", "Não")

display(df_vendas_gerentes_janeiro_23)

In [ ]:
df_analise["Mes-Ano"] = df_analise["Data"].dt.to_period("M")
df_vendas_mes = df_analise[["Mes-Ano", "Faturamento"]].groupby("Mes-Ano").sum()
df_vendas_mes.plot()

display(df_vendas_mes)